# 01 — Bronze Layer Raw Data Storage (Databricks PySpark & Delta Lake)
- **Workstream 2:** Lakehouse & Data Quality (Member 2 — Ahmed Essam)
- **ClickUp Task:** `123u8tync4t` (*Implement Bronze raw-data storage*)
- **Sources Ingested:**
  1. `SRC-001`: openFDA Drug NDC (`openfda_ndc_20261102T155044Z.json` — 100 records)
  2. `SRC-002`: openFDA Drug Enforcement Recalls (`openfda_enforcement_20261102T155044Z.json` — 100 records)
  3. `SRC-007`: WHO Essential Medicines List 2023 (`who_eml_raw.json` — 69 pages)

In [ ]:
from pyspark.sql import functions as F

VOLUME_PATH = "/Volumes/workspace/default/raw_data"

# 1. Bronze openFDA NDC Table
raw_ndc = spark.read.option("multiLine", "true").json(
    f"{VOLUME_PATH}/openfda_ndc_20261102T155044Z.json"
)

bronze_ndc_df = (
    raw_ndc.select(
        F.col("metadata.source_id").alias("source_id"),
        F.col("metadata.source_name").alias("source_name"),
        F.col("metadata.source_endpoint").alias("source_endpoint"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.col("metadata.run_id").alias("run_id"),
        F.explode("data.results").alias("drug")
    )
    .select("source_id", "source_name", "source_endpoint", "ingestion_timestamp_utc", "run_id", "drug.*")
    .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

bronze_ndc_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_openfda_ndc")
print(f"bronze_openfda_ndc rows: {bronze_ndc_df.count()}")
display(bronze_ndc_df.limit(5))

In [ ]:
# 2. Bronze openFDA Enforcement Recalls Table
raw_enf = spark.read.option("multiLine", "true").json(
    f"{VOLUME_PATH}/openfda_enforcement_20261102T155044Z.json"
)

bronze_enf_df = (
    raw_enf.select(
        F.col("metadata.source_id").alias("source_id"),
        F.col("metadata.source_name").alias("source_name"),
        F.col("metadata.source_endpoint").alias("source_endpoint"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.col("metadata.run_id").alias("run_id"),
        F.explode("data.results").alias("recall")
    )
    .select("source_id", "source_name", "source_endpoint", "ingestion_timestamp_utc", "run_id", "recall.*")
    .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

bronze_enf_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_openfda_enforcement")
print(f"bronze_openfda_enforcement rows: {bronze_enf_df.count()}")
display(bronze_enf_df.limit(5))

In [ ]:
# 3. Bronze WHO Essential Medicines List (2023) Table
raw_who = spark.read.option("multiLine", "true").json(
    f"{VOLUME_PATH}/who_eml_raw.json"
)

bronze_who_df = (
    raw_who.select(
        F.lit("SRC-007").alias("source_id"),
        F.col("metadata.source").alias("source_name"),
        F.col("metadata.source_file").alias("source_file"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.explode("results").alias("page_rec")
    )
    .select("source_id", "source_name", "source_file", "ingestion_timestamp_utc", "page_rec.*")
    .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

bronze_who_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_who_eml")
print(f"bronze_who_eml rows: {bronze_who_df.count()}")
display(bronze_who_df.limit(5))